# Quantum teleportation

Quantum teleportation moves the state of one qubit to another, using entanglement and classical communication, without moving matter or transmitting anything faster than light: only the state is transmitted, and the classical channel is needed to complete the process. This notebook builds a minimal version with three qubits.

## The protocol, in words

Alice has a qubit with the state she wants to send to Bob. Alice and Bob share a Bell pair beforehand, one qubit each. Alice combines her qubit with her half of the Bell pair, measures both, and sends those classical bits to Bob. With that information, Bob applies a correction to his own qubit, X, Z, neither, or both, depending on the bits received, and his qubit ends up with the original state.

## Why this version is different

The textbook protocol measures two qubits partway through the circuit and applies a correction conditioned on the result. Polypus does not support that: it only allows measuring at the end, with `measure` or `measure_all`, with no classical conditionals on a measurement result.

The solution is to apply the corrections as controlled quantum gates, before measuring, instead of as classical gates conditioned after measuring. By the principle of deferred measurement, both versions give the same physical result: a CX or CZ controlled by a qubit behaves the same as measuring that qubit and applying an X or Z depending on the outcome.

## Building the circuit

Qubit 0 holds the state to send, qubit 1 is Alice's half of the Bell pair, and qubit 2 is Bob's half, where the state should end up:

In [ ]:
import math

import polypus

theta = math.pi / 3

qc = polypus.Circuit(3)
qc.rx(0, theta)  # state to teleport, on qubit 0
qc.h(1)  # Bell pair between qubits 1 and 2
qc.cx(1, 2)
qc.cx(0, 1)  # Alice combines her qubit with her half of the Bell pair
qc.h(0)
qc.cx(1, 2)  # deferred X correction
qc.cz(0, 2)  # deferred Z correction
qc.measure_all()

With the pattern seen in [`02b`](02b_qiskit_qasm_interop.ipynb), the full circuit can be seen:

In [ ]:
from qiskit import qasm2

qc_drawing = qasm2.loads(
    qc.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_drawing.draw("mpl")

## Checking the result

With `rx(0, theta)`, the theoretical probability of measuring qubit 0 as `1` is `sin²(theta/2)`. If teleportation works, qubit 2 should reproduce that same probability:

In [ ]:
result = polypus.run_quantum_circuit(qc, shots=10000, infrastructure="local")
counts = result.counts[0]

total = sum(counts.values())
p1_qubit2 = sum(c for bits, c in counts.items() if bits[0] == "1") / total

print("measured:", p1_qubit2)
print("theoretical:", math.sin(theta / 2) ** 2)

The first character of the measured string corresponds to the last qubit of the circuit, qubit 2 in this case: Polypus orders the measured bits from highest to lowest qubit index, left to right. The two values agree within the margin expected from sampling noise: the state has genuinely moved from qubit 0 to qubit 2.

## Summary

This notebook built quantum teleportation with three qubits, replacing the textbook protocol's mid-circuit measurement and classical correction with the equivalent controlled gates before measuring, and checked that the state arrives intact at the destination qubit by comparing its probability with the theoretical value.

The next section covers another classic construction: the quantum Fourier transform.

## Next step

Continue with: [`04b_qft.ipynb`](04b_qft.ipynb).